# ScoreNN: score prepared interactions

Apply ScoreNN to interaction features you have already prepared.

This notebook previews the configuration before running your data. Running every cell with the default settings does not start a prediction.


## 1. Before you start

Install MaSIF using the [README installation instructions](../README.md#installation). Use a notebook kernel from your `masif_torch` environment, and obtain the checkpoint listed in the [weights guide](../weights/README.md).

Follow [Open a tutorial](README.md#open-a-tutorial) to install Jupyter and register the `masif_torch` kernel. Confirm the selected notebook kernel uses the same Python as your MaSIF installation.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Locate the checkout whether this notebook starts in tutorials/ or the project root.
start = Path.cwd().resolve()
project_root = next(
    (path for path in (start, *start.parents)
     if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch checkout.")
os.chdir(project_root)
print("Project:", project_root)
print("Notebook Python:", sys.executable)


## 2. Prepare your input

Provide an NPZ packet with prepared **ScoreNN interaction features** under the `features` key. This task expects features prepared for ScoreNN; it does not take a raw PDB structure or a surface packet.

The checkpoint below belongs to ScoreNN. For the IPA scorer used by MaSIF-seed and MaSIF-neosurface, use the [alignment tutorial](alignment.ipynb) instead.

If you want to start from target and candidate surfaces, follow the [seed](seed.ipynb) or [neosurface](neosurface.ipynb) workflow.


## 3. Choose paths and settings

Replace `input_path` with your prepared packet. Choose an output path for this run. Keep `device = "cpu"` for a CPU run, or select `"cuda"` after installing a compatible GPU runtime.

Leave `RUN_TASK = False` while checking the configuration. Set it to `True` only when the input and checkpoint are ready.


In [ ]:
checkpoint = Path("weights/score/second_stage_interface_scoring.pt")
input_path = Path("data/score/example.npz")
output_path = Path("results/score/example.npy")
device = "cpu"
batch_size = 100
RUN_TASK = False

command = [
    sys.executable, "-m", "masif", "score",
    "--config", "examples/score.yaml",
    f"checkpoint={checkpoint}",
    f"input={input_path}",
    f"output={output_path}",
    f"device={device}",
    f"batch_size={batch_size}",
]


## 4. Preview the configuration

This step checks the command and displays your settings without reading the input or loading a checkpoint. Review the paths and device before continuing.


In [ ]:
print(shlex.join(command))
preview = subprocess.run(command + ["--cfg", "job"], check=True, capture_output=True, text=True)
print(preview.stdout)


## 5. Run your prepared input

After updating the settings, set `RUN_TASK = True` and rerun the settings, preview and run cells. The run cell checks that the input and checkpoint exist before starting.


In [ ]:
if RUN_TASK:
    for required_path in (input_path, checkpoint):
        if not required_path.is_file():
            raise FileNotFoundError(f"Required file not found: {required_path}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(command, check=True)
    print("Saved:", output_path)
else:
    print("Preview only. Set RUN_TASK = True after preparing your files.")


## 6. Next steps

To fit ScoreNN to labeled interaction packets, see the [training tutorial](training.ipynb) and [train_score.yaml](../examples/train_score.yaml).

For the complete surface search and alignment workflow, continue with [MaSIF-seed](seed.ipynb).

See the [usage manual](../USAGE.md) for other commands and configuration examples.
